# D-Wave Ocean SDK Framework with Simulated Annealing

This tutorial outlines how to use the D-Wave Ocean SDK to formulate and solve a QUBO optimization problem using simulated annealing. Simulated annealing runs entirely on the local machine, so no access to the D-Wave Leap quantum cloud service is required.

**If you do have access to the Leap quantum cloud service then check out our tutorial on [solving QUBO problems using a quantum annealer](https://kaufmann-group.github.io/qstep-demo/D-wave%20SDK%20Demonstration%20Quantum%20Annealing.html).**

## Installation & Setup



In this section, we outline how to install the D-Wave Ocean SDK. It is recommended that the Ocean SDK be installed inside a Python virtual environment rather than directly into the system as virtual environments keep the packages used for this project isolated from other Python projects and avoid potential dependency conflicts.

The virtual environment can also be registered as a Jupyter kernel, allowing a Jupyter notebook to run directly inside the same environment in which the Ocean SDK is installed. To create a virtual environment and activate it in macOS or Linux run,

``` bash
python3 -m venv ocean
source ocean/bin/activate
``` 

To create and run a virtual environment in Windows' command shell run,

``` bat
python -m venv ocean
ocean\Scripts\activate.bat
```

Once the environment is activated, any Python packages installed with `pip` will be installed inside the ocean environment rather than globally. To install the Ocean SDK and other libraries that will be used run,

``` bash
python -m pip install --upgrade pip

python -m pip install dwave-ocean-sdk
python -m pip install yfinance 
```

Since this tutorial uses Jupyter notebooks we must register the environment with Jupyter,

``` bash
python -m pip install ipykernel
python -m ipykernel install --user --name ocean --display-name "Ocean Environment"
```

**Note after opening the Jupyter Notebook make sure to select the "Ocean Environment" kernel.**

## Review

Recall that quantum annealers are designed to solve certain classes of optimization problems, one of the most common being [Quadratic Unconstrained Binary Optimization (QUBO)](https://en.wikipedia.org/wiki/Quadratic_unconstrained_binary_optimization) problems. D-Wave's Quantum Processing Unit (QPU) represents these problems using an Ising Hamiltonian of the form

$$H(\mathbf{s}) = \sum_i h_i s_i + \sum_{i<j} J_{ij}s_i s_j,\qquad s_i\in\{-1,+1\},$$

where the coefficients $h_i$ represent individual biases and $J_{ij}$ represent interactions between pairs of spins. Using a simple change of variables between Ising spins and binary variables, typically

$$s_i = 2x_i-1,$$

we can rewrite the same problem as a QUBO,

$$Q(\mathbf{x}) = \sum_i a_i x_i + \sum_{i<j} b_{ij}x_i x_j, \qquad x_i\in\{0,1\}.$$

This allows an optimization problem written as a QUBO to be mapped directly onto the Ising model implemented by the D-Wave hardware. The QPU uses a network of superconducting qubits whose interactions are programmed according to the coefficients of the problem. During quantum annealing, the system is driven toward low-energy configurations of the Ising Hamiltonian. After measurement, these spin configurations can be converted back into binary variables, with the lowest-energy solutions corresponding to minima of the original QUBO objective function. Learn more about the physics of quantum annealing in [this]() notebook!

## Optimization Problem Statement: Markowitz Portfolio Optimization

<div style="display: flex; align-items: center; gap: 30px;">
<div style="flex: 2;">
<p>
A common financial optimization problem is known as the Markowitz mean variance portfolio optimization. It states that given a set of assets with known expected returns and correlations, the goal is to determine how much of a portfolio should be allocated to each asset.
    
It states that an optimal portfolio balances two competing objectives: maximizing expected return while minimizing financial risk. The set of portfolios that provide the best possible return for a given level of risk forms what is known as the efficient frontier.
</p>
</div>

<div style="flex: 1; text-align: center;">
<img src="./images/efficient-frontier.jpg" style="width: 100%; max-width: 300px; height: auto;">
</div>
</div>

Say we have a collection of assets in a portfolio and the variable,

$$\mathbf{w} = \begin{pmatrix} w_1 \\ w_2 \\ \vdots \\ w_N \end{pmatrix}$$

represents the fraction of the portfolio allocated to each asset, where $\mu_i$ is the [expected return](https://en.wikipedia.org/wiki/Expected_return) of asset $i$ and $\Sigma$ is the [covariance matrix](https://en.wikipedia.org/wiki/Covariance_matrix) describing the risk and correlations between assets. The expected of the portfolio is given by the expression,

$$\boldsymbol{\mu}^{T}\mathbf{w},$$

while the portfolio variance, which is used as a [measure of risk](https://en.wikipedia.org/wiki/Risk_measure), is given by the expression,

$$\mathbf{w}^{T}\Sigma\mathbf{w}.$$

The optimization problem can therefore be written as

$$\min_{\mathbf{w}} \left[ \lambda_1 \mathbf{w}^{T}\Sigma\mathbf{w} - \lambda_2 \boldsymbol{\mu}^{T}\mathbf{w} \right],$$

where the first term penalizes portfolio risk and the second term rewards expected return. In this optimization problem we also require the entire portfolio to be allocated, or simply put the weights must sum to $1$.

$$\sum_i w_i = 1$$

Because a QUBO is an unconstrained optimization problem, this condition can be included using a quadratic penalty,

$$\lambda_3 \left( \sum_i w_i - 1 \right)^2.$$

Therefore complete objective function we seek to mimize becomes,

$$H(\mathbf{w}) = \lambda_3 \mathbf{w}^{T}\Sigma\mathbf{w} - \lambda_1 \boldsymbol{\mu}^{T}\mathbf{w} + \lambda_2 \left( \sum_i w_i - 1 \right)^2.$$

Here, $\lambda_1$ controls the importance of expected return, $\lambda_2$ controls the penalty associated with portfolio risk and  $\lambda_3$ controls the portfolio normalization constraint.

Something worth noting is that the portfolio weights $w_i$ are continuous variables, while a QUBO requires binary variables. So to represent each portfolio weight using several binary variables we look at its binary expansion

$$w_i = \sum_{k=1}^{b} 2^{-k}x_{ik}, \quad x_{ik}\in\{0,1\}.$$

Substituting these binary representations into the objective converts the portfolio optimization problem into a quadratic function of binary variables,

$$H(\mathbf{x})=\mathbf{x}^{T}Q\mathbf{x}.$$


## Yahoo Finiance Data

For this demonstration we get the average returns from Yahoo Finiance from the `yfinance` library (ensure it is installed in this virtual environment). A full list of the assets accessable are given [here](assets.txt). 

In [1]:
import yfinance as yf

assets = ["AAPL", "MSFT", "NVDA", "AMD", "CSCO", "ORCL", "QCOM", "IBM"]

start="2025-04-04"
end="2026-04-04"

data = yf.download(assets, start=start, end=end)
closing_prices = data["Close"].pct_change().dropna()

# The 252 represents the number of business days in a year
returns = closing_prices.mean() * 252 
covariance = closing_prices.cov() * 252

[*********************100%***********************]  8 of 8 completed


## Using Ocean's Simulated Annealing Sampler

The 

In [2]:
from collections import defaultdict

Q = defaultdict(float)

lambda_1 = 1.0    # Expected return
lambda_2 = 10.0   # Budget constraint
lambda_3 = 1.0    # Portfolio risk

bits_per_asset = 6

# Binary coefficients used to represent each portfolio weight
bit_weights = [2**k / (2**bits_per_asset - 1) for k in range(bits_per_asset)]


variables = []

for asset in assets:
    for k in range(bits_per_asset):
        variable = f"{asset}_{k}"
        coefficient = bit_weights[k]

        variables.append((variable, asset, coefficient))

for variable, asset, coefficient in variables:
    mu_i = returns.loc[asset]
    sigma_ii = covariance.loc[asset, asset]

    Q[(variable, variable)] += (lambda_3 * sigma_ii * coefficient**2 - lambda_1 * mu_i * coefficient + lambda_2 * (coefficient**2 - 2 * coefficient))

for a in range(len(variables)):
    variable_i, asset_i, coefficient_i = variables[a]

    for b in range(a + 1, len(variables)):

        variable_j, asset_j, coefficient_j = variables[b]
        sigma_ij = covariance.loc[asset_i, asset_j]

        Q[(variable_i, variable_j)] += (2 * coefficient_i * coefficient_j * ( lambda_3 * sigma_ij + lambda_2 ))


Q = dict(Q)

Because simulated annealing runs on a classical computer rather than on a QPU, there is no hardware graph to map the problem onto and no minor embedding step. Any QUBO can be passed to the sampler directly, regardless of how its variables are connected.

Simulated annealing mimics the physical annealing process. Each run starts from a random configuration at a high effective temperature, where moves that increase the energy are often accepted, and the temperature is then lowered over a series of sweeps so that the search gradually settles into a low-energy configuration. Ocean provides this through `SimulatedAnnealingSampler`, which is part of the `dwave-samplers` package included in the Ocean SDK.

In [3]:
from dwave.samplers import SimulatedAnnealingSampler

sampler = SimulatedAnnealingSampler()

sampleset = sampler.sample_qubo(Q, num_reads=1000)

best = sampleset.first # returns lowest energy sample in the SampleSet
solution = dict(best.sample)  

print("Solution:", solution)
print("Energy:", best.energy)

Solution: {'AAPL_0': np.int8(0), 'AAPL_1': np.int8(0), 'AAPL_2': np.int8(0), 'AAPL_3': np.int8(0), 'AAPL_4': np.int8(0), 'AAPL_5': np.int8(0), 'AMD_0': np.int8(1), 'AMD_1': np.int8(1), 'AMD_2': np.int8(0), 'AMD_3': np.int8(0), 'AMD_4': np.int8(1), 'AMD_5': np.int8(1), 'CSCO_0': np.int8(0), 'CSCO_1': np.int8(0), 'CSCO_2': np.int8(0), 'CSCO_3': np.int8(0), 'CSCO_4': np.int8(0), 'CSCO_5': np.int8(0), 'IBM_0': np.int8(0), 'IBM_1': np.int8(0), 'IBM_2': np.int8(0), 'IBM_3': np.int8(0), 'IBM_4': np.int8(0), 'IBM_5': np.int8(0), 'MSFT_0': np.int8(0), 'MSFT_1': np.int8(0), 'MSFT_2': np.int8(0), 'MSFT_3': np.int8(0), 'MSFT_4': np.int8(0), 'MSFT_5': np.int8(0), 'NVDA_0': np.int8(1), 'NVDA_1': np.int8(0), 'NVDA_2': np.int8(1), 'NVDA_3': np.int8(1), 'NVDA_4': np.int8(0), 'NVDA_5': np.int8(0), 'ORCL_0': np.int8(0), 'ORCL_1': np.int8(0), 'ORCL_2': np.int8(0), 'ORCL_3': np.int8(0), 'ORCL_4': np.int8(0), 'ORCL_5': np.int8(0), 'QCOM_0': np.int8(0), 'QCOM_1': np.int8(0), 'QCOM_2': np.int8(0), 'QCOM_3': n

Recall that `solution` is still in its binary representation, where each vertex has been assigned either `0` or `1`; so it follows to separate the vertices into the two groups.

In [4]:
portfolio = {}

for asset in assets:

    portfolio[asset] = sum(bit_weights[k] * solution[f"{asset}_{k}"] for k in range(bits_per_asset))

print(portfolio)

{'AAPL': np.float64(0.0), 'MSFT': np.float64(0.0), 'NVDA': np.float64(0.20634920634920634), 'AMD': np.float64(0.8095238095238095), 'CSCO': np.float64(0.0), 'ORCL': np.float64(0.0), 'QCOM': np.float64(0.0), 'IBM': np.float64(0.0)}


Finished? Check out our other tutorials! 

<a class="notebook-download"
   href="https://github.com/kaufmann-group/qstep-demo/raw/refs/heads/main/D-wave%20Simulated%20Annealing%20Demonstration.ipynb"
   download>
   Download notebook ↓
</a>